# Aggregation and reporting

Grouping and summarising, which is the shape of most dashboards and daily metrics.

Compare **A1** against **A2**. A1 groups into four buckets, so almost nothing moves between
workers. A2 groups by customer, producing millions of buckets and forcing a real shuffle. That
is Spark's home ground, and the gap should narrow.

**A5** and **A6** are marked approximate. Both engines answer them with sketching algorithms
that are inexact and implemented differently, so the answers legitimately differ.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))
from bench import config as C, datagen, engines, report
from bench.harness import Case, Bench

paths = datagen.paths(C.MAIN_SIZE)
duck  = engines.get_duckdb()
spark = engines.get_spark()
engines.attach(duck, spark, paths)
bench = Bench(duck, spark, "02_aggregation", C.MAIN_SIZE)
print(f"Ready. {C.human(C.MAIN_SIZE)} sales rows. "
      f"Both engines have {C.ENGINE_MEMORY_MB} MB and {C.plural(C.ENGINE_THREADS, 'thread')}.")


In [ ]:
SQL_A1 = """
SELECT channel, count(*) AS orders, round(sum(amount),2) AS revenue,
       round(avg(amount),2) AS avg_order
FROM sales GROUP BY 1 ORDER BY revenue DESC
"""

_, out, _ = bench.run(Case("A1", "Group by 4 values (low cardinality)", "Aggregation", sql=SQL_A1.strip()))
display(out.head())


In [ ]:
SQL_A2 = """
SELECT customer_id, count(*) AS orders, round(sum(amount),2) AS revenue
FROM sales WHERE customer_id IS NOT NULL
GROUP BY 1 HAVING count(*) > 3 ORDER BY revenue DESC LIMIT 50
"""

_, out, _ = bench.run(Case("A2", "Group by customer (high cardinality)", "Aggregation", sql=SQL_A2.strip()))
display(out.head())


In [ ]:
SQL_A3 = """
SELECT region, channel, count(*) AS orders, round(sum(amount),2) AS revenue
FROM sales GROUP BY ROLLUP (region, channel) ORDER BY region, channel
"""

_, out, _ = bench.run(Case("A3", "Multi-level rollup", "Aggregation", sql=SQL_A3.strip()))
display(out.head())


In [ ]:
SQL_A4 = """
SELECT count(DISTINCT customer_id) AS customers, count(DISTINCT product_id) AS products,
       count(DISTINCT store_id) AS stores
FROM sales
"""

_, out, _ = bench.run(Case("A4", "Exact distinct count", "Aggregation", sql=SQL_A4.strip()))
display(out.head())


In [ ]:
SQL_A5 = """
SELECT approx_count_distinct(customer_id) AS customers,
       approx_count_distinct(session_ref) AS sessions
FROM sales
"""

_, out, _ = bench.run(Case("A5", "Approximate distinct count", "Aggregation", sql=SQL_A5.strip(), approximate=True))
display(out.head())


In [ ]:
SQL_A6_DUCK = """
SELECT round(quantile_cont(amount, 0.5),2) AS p50, round(quantile_cont(amount, 0.95),2) AS p95 FROM sales WHERE amount IS NOT NULL
"""

SQL_A6_SPARK = """
SELECT round(percentile_approx(amount, 0.5),2) AS p50, round(percentile_approx(amount, 0.95),2) AS p95 FROM sales WHERE amount IS NOT NULL
"""

_, out, _ = bench.run(Case("A6", "Percentiles", "Aggregation", duck_sql=SQL_A6_DUCK.strip(), spark_sql=SQL_A6_SPARK.strip(), approximate=True))
display(out.head())


In [ ]:
SQL_A7 = """
SELECT category, product_id, revenue FROM (
  SELECT p.category, s.product_id, round(sum(s.amount),2) AS revenue,
         row_number() OVER (PARTITION BY p.category ORDER BY sum(s.amount) DESC) AS rk
  FROM sales s JOIN products p ON p.product_id = s.product_id
  GROUP BY 1, 2
) t WHERE rk <= 3 ORDER BY category, revenue DESC
"""

_, out, _ = bench.run(Case("A7", "Top 3 products per category", "Aggregation", sql=SQL_A7.strip()))
display(out.head())


In [ ]:
SQL_A8 = """
SELECT round(avg(amount),3) AS mean, round(stddev(amount),3) AS sd,
       round(min(amount),2) AS lo, round(max(amount),2) AS hi,
       round(corr(quantity, amount),4) AS corr_qty_amt
FROM sales WHERE amount IS NOT NULL
"""

_, out, _ = bench.run(Case("A8", "Statistical summary", "Aggregation", sql=SQL_A8.strip()))
display(out.head())


## Results for this notebook

`Same SQL?` tells you whether both engines ran the *identical* SQL string. Where it says no, the two dialects genuinely differ and the case is written twice.

`Same answer?` is the check that matters: a fast wrong answer is worth nothing.

In [ ]:
report.headline(bench.table())
print()
display(report.results_table(bench.table()))
report.times_chart(bench.table())
bench.save()
engines.stop_spark()
